# XGBoost: Six-Class vs Three-Class Next-Day AQI

Compare XGBoost for six-class and three-class next-day AQI prediction using the same daily data, feature set, chronological holdout, and date-based cross-validation folds.

The notebook reads `air_quality.aqi_daily` only. It does not write to MySQL or save files.

## Evaluation design

- Rebuild AQI and next-calendar-day targets with the shared AQI labeling rules.
- Use one chronological split by unique date: the first 80% for training and the last 20% for the untouched test period. All stations for a date stay in the same split.
- Tune XGBoost only on the training dates using four expanding-window date-based folds and macro-F1.
- Compare accuracy, macro-F1, weighted F1, and balanced accuracy with majority and persistence baselines.
- Review per-class precision, recall, F1, support, and confusion matrices.

The target is the next day's AQI class. The six-class task preserves all AQI categories; the three-class task groups them into clean/good, light-to-moderate, and heavily polluted.

## 1. Imports and project settings

The notebook locates the shared AQI helper and reads from the configured MySQL `air_quality` database. Set `AIR_QUALITY_DB_URL` or `DB_URL` if needed.

In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sqlalchemy import create_engine, text
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, make_scorer, precision_recall_fscore_support,
)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

HERE = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [HERE, *HERE.parents]
     if (p / "air-quality-classification" / "app" / "aqi_labels.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the Data_Science repository or one of its subfolders.")
APP_DIR = PROJECT_ROOT / "air-quality-classification" / "app"
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))
from aqi_labels import O3_1H_BREAKPOINTS, POLLUTANT_BREAKPOINTS, AQI_CATEGORIES, category_codes, interpolate_iaqi

DB_URL = os.environ.get(
    "AIR_QUALITY_DB_URL",
    os.environ.get("DB_URL", "mysql+pymysql://root:@localhost/air_quality"),
)
SOURCE_TABLE = "aqi_daily"
POLLUTANTS = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]
WEATHER = [
    "TEMP_mean", "TEMP_range", "PRES_mean", "DEWP_mean", "RH_mean",
    "RAIN_sum", "WSPM_mean", "wd_sin", "wd_cos",
]
FULL_FEATURES = POLLUTANTS + WEATHER
RANDOM_STATE = 42
PARAM_GRID = {
    "classifier__n_estimators": [120, 240],
    "classifier__max_depth": [3, 5],
    "classifier__learning_rate": [0.06],
}

print("Project:", PROJECT_ROOT)
print("Read-only source:", "air_quality." + SOURCE_TABLE)
print("Full feature count:", len(FULL_FEATURES))

## 2. Read and validate `aqi_daily`

This cell performs a read-only query, checks required columns, and verifies one row per station and date.

In [ ]:
required = ["date", "station", "station_id", *POLLUTANTS, "O3_1h_max", *WEATHER]
engine = create_engine(DB_URL, pool_pre_ping=True)
try:
    with engine.connect() as connection:
        source = pd.read_sql_query(text(f"SELECT * FROM {SOURCE_TABLE}"), connection)
finally:
    engine.dispose()

missing = [column for column in required if column not in source.columns]
if missing:
    raise KeyError(f"{SOURCE_TABLE} is missing required columns: {missing}")
source["date"] = pd.to_datetime(source["date"], errors="coerce").dt.normalize()
if source["date"].isna().any():
    raise ValueError("aqi_daily contains invalid dates.")
if source.duplicated(["station_id", "date"]).any():
    raise ValueError("aqi_daily has duplicate station-date keys.")
for column in [*POLLUTANTS, "O3_1h_max", *WEATHER]:
    source[column] = pd.to_numeric(source[column], errors="coerce")

print(f"Rows: {len(source):,}; stations: {source.station.nunique():,}")
print(f"Date range: {source.date.min().date()} through {source.date.max().date()}")
display(source.head())

## 3. Rebuild AQI and exact next-day targets

AQI uses the shared pollutant breakpoints. Tomorrow's target is joined by the exact next calendar date for the same station; gaps are not treated as consecutive days.

In [ ]:
daily = source.copy()
iaqi_columns = []
for pollutant in POLLUTANTS:
    values = daily[pollutant]
    iaqi_column = f"{pollutant}_iaqi"
    if pollutant == "O3":
        ozone_8h = interpolate_iaqi(values, POLLUTANT_BREAKPOINTS["O3"])
        ozone_1h = interpolate_iaqi(daily["O3_1h_max"], O3_1H_BREAKPOINTS)
        use_one_hour = values.gt(800)
        ozone_8h.loc[use_one_hour] = ozone_1h.loc[use_one_hour]
        daily[iaqi_column] = ozone_8h.round()
    else:
        daily[iaqi_column] = interpolate_iaqi(
            values, POLLUTANT_BREAKPOINTS[pollutant]
        ).round()
    iaqi_columns.append(iaqi_column)

daily["AQI"] = daily[iaqi_columns].max(axis=1, skipna=True)
daily["category_code"] = category_codes(daily["AQI"])
today = daily[["date", "station_id", "station", *FULL_FEATURES, "AQI", "category_code"]].copy()
tomorrow = daily[["date", "station_id", "AQI", "category_code"]].copy()
tomorrow["date"] = tomorrow["date"] - pd.Timedelta(days=1)
tomorrow = tomorrow.rename(columns={
    "AQI": "target_AQI", "category_code": "target_category_code"
})
data = today.merge(
    tomorrow, on=["date", "station_id"], how="inner", validate="one_to_one"
).dropna(subset=["target_AQI", "target_category_code"]).copy()
data["target_6class"] = data["target_category_code"].astype(int) - 1
data["today_6class"] = data["category_code"].astype(float) - 1
data["target_3class"] = np.select(
    [data["target_AQI"] <= 100, data["target_AQI"] <= 200], [0, 1], default=2
).astype(int)
data["today_3class"] = np.select(
    [data["AQI"] <= 100, data["AQI"] <= 200], [0, 1], default=2
).astype(float)
data.loc[data["AQI"].isna(), "today_3class"] = np.nan
data = data.sort_values(["date", "station"]).reset_index(drop=True)

CLASS_NAMES = {
    "6-class": [AQI_CATEGORIES[i] for i in range(1, 7)],
    "3-class": ["Clean / good", "Light-to-moderate polluted", "Heavily polluted"],
}
TASKS = {
    "6-class": {"target": "target_6class", "today": "today_6class", "labels": list(range(6))},
    "3-class": {"target": "target_3class", "today": "today_3class", "labels": list(range(3))},
}
print(f"Exact next-calendar-day examples: {len(data):,}")
for task, spec in TASKS.items():
    counts = data[spec["target"]].value_counts().reindex(spec["labels"], fill_value=0)
    display(pd.DataFrame({"class": CLASS_NAMES[task], "rows": counts.to_numpy()}))

## 4. Chronological train/test split and date-based folds

The same held-out dates and expanding-window folds are used in all experiments in this notebook.

In [ ]:
unique_dates = np.array(sorted(data["date"].dropna().unique()))
if len(unique_dates) < 10:
    raise ValueError("Not enough unique dates for the chronological holdout and four time-series folds.")
cut = int(0.80 * len(unique_dates))
train_dates, test_dates = unique_dates[:cut], unique_dates[cut:]
train = data[data["date"].isin(train_dates)].copy().reset_index(drop=True)
test = data[data["date"].isin(test_dates)].copy().reset_index(drop=True)

folds = []
for earlier, later in TimeSeriesSplit(n_splits=4).split(train_dates):
    earlier_dates, later_dates = train_dates[earlier], train_dates[later]
    folds.append((
        np.flatnonzero(train["date"].isin(earlier_dates)),
        np.flatnonzero(train["date"].isin(later_dates)),
    ))
print(f"Training: {len(train):,} rows, {train.date.min().date()} to {train.date.max().date()}")
print(f"Test: {len(test):,} rows, {test.date.min().date()} to {test.date.max().date()}")
print(f"One shared chronological holdout; {len(folds)} expanding date-based folds.")

## 5. Define XGBoost experiment

Both models use today's six pollutant summaries and nine daily weather summaries. The only difference is the number of target categories.

In [ ]:
def build_xgb_pipeline(n_classes):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", XGBClassifier(
            objective="multi:softprob", num_class=n_classes,
            eval_metric="mlogloss", random_state=RANDOM_STATE,
            n_jobs=4, tree_method="hist", verbosity=0,
        )),
    ])

def metric_row(task, model_name, y_true, prediction, cv_macro_f1=np.nan, best_params=None):
    labels = TASKS[task]["labels"]
    return {
        "task": task,
        "model": model_name,
        "cv_macro_f1": cv_macro_f1,
        "accuracy": accuracy_score(y_true, prediction),
        "macro_f1": f1_score(y_true, prediction, labels=labels, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, prediction, labels=labels, average="weighted", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
        "best_params": best_params if best_params is not None else "-",
    }

## 6. Tune and evaluate XGBoost for both class schemes

Each task gets its own multiclass XGBoost model and macro-F1 parameter search. Both use the same pollutant-plus-weather inputs and same date folds.

In [ ]:
TASKS = {
    "6-class": {"target": "target_6class", "today": "today_6class", "labels": list(range(6))},
    "3-class": {"target": "target_3class", "today": "today_3class", "labels": list(range(3))},
}
models, predictions, result_rows = {}, {}, []
for task, spec in TASKS.items():
    target_col = spec["target"]
    y_train = train[target_col].to_numpy(dtype=int)
    task_test = test.dropna(subset=[target_col]).copy()
    y_test = task_test[target_col].to_numpy(dtype=int)
    n_classes = len(spec["labels"])
    if sorted(np.unique(y_train).tolist()) != spec["labels"]:
        raise ValueError(f"{task}: training dates do not contain every expected class.")
    scorer = make_scorer(
        f1_score, labels=spec["labels"], average="macro", zero_division=0
    )
    search = GridSearchCV(
        build_xgb_pipeline(n_classes), PARAM_GRID, scoring=scorer,
        cv=folds, n_jobs=1, refit=True, return_train_score=False,
    )
    started = time.time()
    search.fit(train[FULL_FEATURES], y_train)
    prediction = np.asarray(search.best_estimator_.predict_proba(task_test[FULL_FEATURES])).argmax(axis=1)
    models[task] = search.best_estimator_
    predictions[task] = np.full(len(test), np.nan)
    predictions[task][test[target_col].notna().to_numpy()] = prediction
    result_rows.append(metric_row(task, "XGBoost", y_test, prediction,
                                  search.best_score_, search.best_params_))
    print(f"{task}: CV macro-F1={search.best_score_:.4f}; test accuracy={accuracy_score(y_test, prediction):.4f}; elapsed={time.time()-started:.1f}s")
    print("Best parameters:", search.best_params_)

## 7. Baselines and aggregate metrics

Majority always predicts the most common training class. Persistence repeats today's class for tomorrow, using the training majority when today's AQI is unavailable.

In [ ]:
# Add training-majority and persistence baselines for each target definition.
majority_predictions = {}
for task, spec in TASKS.items():
    target_col, today_col, labels = spec["target"], spec["today"], spec["labels"]
    majority_class = int(train[target_col].value_counts().idxmax())
    majority_prediction = np.full(len(test), majority_class, dtype=int)
    persistence_prediction = test[today_col].fillna(majority_class).to_numpy(dtype=int)
    majority_predictions[(task, "Majority baseline")] = majority_prediction
    majority_predictions[(task, "Persistence baseline")] = persistence_prediction
    task_mask = test[target_col].notna().to_numpy()
    y_test = test.loc[task_mask, target_col].to_numpy(dtype=int)
    result_rows.extend([
        metric_row(task, "Majority baseline", y_test, majority_prediction[task_mask]),
        metric_row(task, "Persistence baseline", y_test, persistence_prediction[task_mask]),
    ])

results = pd.DataFrame(result_rows)
display(results.sort_values(["task", "macro_f1"], ascending=[True, False]).style.format({
    "cv_macro_f1": "{:.4f}", "accuracy": "{:.2%}", "macro_f1": "{:.4f}",
    "weighted_f1": "{:.4f}", "balanced_accuracy": "{:.2%}",
}, na_rep="-"))

## 8. Per-class reports and confusion matrices

These tables show which classes are difficult and how errors are distributed.

In [ ]:
for task, spec in TASKS.items():
    labels = spec["labels"]
    names = CLASS_NAMES[task]
    task_mask = test[spec["target"]].notna().to_numpy()
    y_true = test.loc[task_mask, spec["target"]].to_numpy(dtype=int)
    print(f"\n{task}: XGBoost per-class report")
    display(pd.DataFrame(classification_report(
        y_true, predictions[task][task_mask].astype(int), labels=labels,
        target_names=names, output_dict=True, zero_division=0,
    )).T.round(4))
    print(f"{task}: XGBoost confusion matrix (rows=actual, columns=predicted)")
    display(pd.DataFrame(
        confusion_matrix(y_true, predictions[task][task_mask], labels=labels),
        index=[f"Actual: {name}" for name in names],
        columns=[f"Predicted: {name}" for name in names],
    ))

## 9. Summary and practical interpretation

The summary below is generated from the newly measured holdout results after the earlier cells run.

In [ ]:
xgb_results = results[results["model"] == "XGBoost"].set_index("task")
base_results = results[results["model"] == "Majority baseline"].set_index("task")
print("XGBoost comparison on the same held-out dates")
for task in ["6-class", "3-class"]:
    row = xgb_results.loc[task]
    baseline = base_results.loc[task]
    print(
        f"{task}: accuracy {row.accuracy:.2%} (majority {baseline.accuracy:.2%}); "
        f"macro-F1 {row.macro_f1:.4f} (majority {baseline.macro_f1:.4f}); "
        f"balanced accuracy {row.balanced_accuracy:.2%}"
    )

print("\nInterpretation")
print("- The 6-class and 3-class scores use different label granularity, so treat each task against its own baselines.")
print("- Use the per-class reports to see whether merging categories improves rare-class recall or mainly raises aggregate accuracy.")
print("- The 3-class model is simpler to communicate; the 6-class model preserves more AQI detail.")
print("- Choose based on the test-period metrics and the app's alert needs, not accuracy alone.")